In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import cvxpy as cp
import warnings
warnings.filterwarnings('ignore')

In [2]:
COLUMNS = {
    'DY-ADJ_AF-OPEN_PRICE_2': 'open',
    'DY-ADJ_AF-HIGHEST_PRICE_2': 'high',
    'DY-ADJ_AF-LOWEST_PRICE_2': 'low',
    'DY-ADJ_AF-CLOSE_PRICE_2': 'close',
    'DY-BASIC-MARKET_VALUE': 'market_cap',

    'LEVEL1_NAME': 'industry',
    'LEVEL2_NAME': 'sub_industry',
    'LEVEL3_NAME': 'sub_sub_industry',
}

In [3]:
def load_data(file_path: str, columns: list[str] = None, rename_map: dict[str, str] = COLUMNS) -> pd.DataFrame:
    """
    载入数据
    """
    df = pd.read_parquet(file_path, columns=columns)

    if file_path == 'industry.parquet':
        df = df.swaplevel('CODE', 'DATE')
        df = df.sort_index()
    elif file_path == 'st.parquet':
        df = df.notna().fillna(False).astype(bool)
        df = df.stack().to_frame('st')
    elif file_path == '停牌.parquet':
        df = df.pivot(index='日期', columns='股票代码', values='是否停牌').fillna(0).astype(bool)
        df = df.stack().to_frame('suspended')

    df.index = df.index.set_names(['date', 'ticker'])
    df.index = df.index.set_levels(pd.to_datetime(df.index.get_level_values('date').unique()), level='date')
    
    df = df.sort_values(['date', 'ticker'])
    df = df.rename(columns=rename_map)
    return df

In [4]:
def check_limits_and_ipo(df: pd.DataFrame, limit_pct=0.10, ipo_days=60) -> tuple[pd.DataFrame]:
    """
    检查涨跌停状态和IPO状态
    """
    og_index = df.index
    df = df.reset_index()

    # 判断涨跌停
    prev_close = df.groupby('ticker')['close'].shift(1)
    limit_up = df['high'] >= prev_close * (1 + limit_pct)
    limit_down = df['low'] <= prev_close * (1 - limit_pct)

    # 判断IPO状态（假设第一个交易日为IPO日）
    first_trade_date = df.groupby('ticker')['date'].transform('min')
    days_since_ipo = (df['date'] - first_trade_date).dt.days
    is_new = days_since_ipo < ipo_days

    limit_up = pd.DataFrame(limit_up.values, index=og_index, columns=['limit_up'])
    limit_down = pd.DataFrame(limit_down.values, index=og_index, columns=['limit_down'])
    is_new = pd.DataFrame(is_new.values, index=og_index, columns=['is_new'])

    return limit_up, limit_down, is_new

In [5]:
def calculate_factor_and_returns(df_ohcl: pd.DataFrame, limit_up: pd.Series, limit_down: pd.Series, df_susp: pd.DataFrame, df_st: pd.DataFrame, is_new: pd.Series) -> pd.DataFrame:
    """
    计算因子值和未来收益率，并剔除不符合条件的股票
    """
    # 因子 =（t0日收盘/t-5日收盘）- 1
    df_ohcl['factor'] = df_ohcl['close'] / df_ohcl.groupby('ticker')['close'].shift(5) - 1

    # 收益率 =（t2日开盘/t1日开盘）- 1
    df_ohcl['factor_returns'] = df_ohcl.groupby('ticker')['open'].transform(lambda x: x.shift(-2) / x.shift(-1) - 1)

    # 剔除涨跌停、停牌、ST、新股
    df_ohcl = df_ohcl[~(limit_up['limit_up'] | limit_down['limit_down'] | df_susp['suspended'] | df_st['st'] | is_new['is_new'])]

    return df_ohcl

In [6]:
def standardize_raw(og_factors: pd.Series, n = 2.5) -> pd.Series:
    """
    因子载荷原始值标准化
    """
    # 计算中位数和MAD
    median = og_factors.median()
    mad = np.median(np.abs(og_factors - median))

    # 标准化
    factors = og_factors.clip(lower=median - n * mad, upper=median + n * mad)
    factors = (factors - factors.mean()) / factors.std()

    return factors

def standardize_rank(og_factors: pd.Series) -> pd.Series:
    """
    因子载荷排序值标准化
    """
    factors = og_factors.rank(method='average')
    factors = (factors - factors.mean()) / factors.std()

    return factors

def standardize_factor(df: pd.DataFrame) -> pd.DataFrame:
    """
    因子值标准化
    """

    df['factor_raw_stdd'] = df.groupby('date')['factor'].transform(standardize_raw)
    df['factor_rank_stdd'] = df.groupby('date')['factor'].transform(standardize_rank)

    return df

In [7]:
def get_residual(y: np.ndarray, X: np.ndarray, index: pd.Index) -> pd.Series:
    """
    获取因子值的残差
    """
    model = sm.OLS(y, X, missing='drop').fit()

    return pd.Series(model.resid, index=index)

def neutralize_factors(df: pd.DataFrame, df_industry: pd.DataFrame, industry_cols: list[str] = ['industry', 'sub_industry', 'sub_sub_industry']) -> pd.DataFrame:
    """
    因子值中性化
    """
    df = df.copy()

    df['log_market_cap'] = np.log(df['market_cap']) # 计算市值的对数

    df = df.merge(df_industry, left_index=True, right_index=True, how='left')

    dummies = pd.get_dummies(df[industry_cols], drop_first=True).astype(float)

    df = pd.concat([df, dummies], axis=1)

    feature_cols = ['log_market_cap'] + list(dummies.columns)

    raw_resids = []
    rank_resids = []
    for date, group in df.groupby(level='date'):
        print(date)
        temp = group.dropna(subset=['factor_raw_stdd', 'factor_rank_stdd', 'log_market_cap'])
        if temp.empty:
            continue

        X = temp[feature_cols].values
        X = sm.add_constant(X)

        y1, y2 = temp['factor_raw_stdd'].values, temp['factor_rank_stdd'].values

        r1, r2 = get_residual(y1, X, temp.index), get_residual(y2, X, temp.index)

        raw_resids.append(r1)
        rank_resids.append(r2)

    df['factor_raw_stdd_neut'] = pd.concat(raw_resids)
    df['factor_rank_stdd_neut'] = pd.concat(rank_resids)

    return df

In [8]:
daily_data = load_data('daily_data.parquet', ['DATE', 'CODE', 'DY-ADJ_AF-OPEN_PRICE_2', 'DY-ADJ_AF-HIGHEST_PRICE_2',
                                                    'DY-ADJ_AF-LOWEST_PRICE_2', 'DY-ADJ_AF-CLOSE_PRICE_2',
                                                    'DY-BASIC-MARKET_VALUE'])
industry_data = load_data('industry.parquet', ['CODE', 'LEVEL1_NAME', 'LEVEL2_NAME', 'LEVEL3_NAME'])
st_data = load_data('st.parquet')
tp_data = load_data('停牌.parquet')

limit_up, limit_down, is_new = check_limits_and_ipo(daily_data)

In [9]:
df_final = calculate_factor_and_returns(daily_data, limit_up, limit_down, tp_data, st_data, is_new)

In [10]:
df_final = standardize_factor(df_final)

In [11]:
df_final

open     high      low    close    market_cap    factor  \
date       ticker                                                               
2010-03-05 000001  953.122  969.214  951.059  960.137  7.226344e+10  0.036526   
           000002  950.561  957.655  942.454  948.534  1.029152e+11 -0.007423   
           000005   58.956   60.823   56.598   60.430  5.623152e+09  0.100167   
           000006  208.253  209.656  206.048  208.253  5.269634e+09  0.000961   
           000009   50.640   51.954   50.158   50.201  1.250000e+10  0.013261   
...                    ...      ...      ...      ...           ...       ...   
2021-12-31 688799   40.020   41.720   40.020   41.170  3.861746e+09  0.080010   
           688800  145.370  149.000  137.500  139.280  1.504224e+10 -0.024650   
           688819   43.126   43.512   43.014   43.430  4.160588e+10  0.019531   
           688981   52.900   53.450   52.740   52.990  4.188254e+11  0.008181   
           689009   69.000   71.250   67.300   70.070  4.960164e+10  0.070425   

                   factor_returns  factor_raw_stdd  factor_rank_stdd  
date       ticker                                                     
2010-03-05 000001        0.024014         1.471805          1.287891  
           000002        0.003206         0.050532          0.187624  
           000005       -0.072381         1.675191          1.651558  
           000006        0.001911         0.321684          0.493383  
           000009        0.009417         0.719442          0.857050  
...                           ...              ...               ...  
2021-12-31 688799             NaN         1.628435          1.400825  
           688800             NaN        -1.296903         -1.285722  
           688819             NaN        -0.062002         -0.074286  
           688981             NaN        -0.379252         -0.508575  
           689009             NaN         1.360504          1.292253  

[7693696 rows x 9 columns]

In [ ]:
df_final = neutralize_factors(df_final, industry_data)

In [ ]:
df_final

open     high      low    close    market_cap    factor  \
date       ticker                                                               
2010-01-04 000001  953.122  969.214  951.059  960.137  7.226344e+10  0.036526   
           000002  950.561  957.655  942.454  948.534  1.029152e+11 -0.007423   
           000005   58.956   60.823   56.598   60.430  5.623152e+09  0.100167   
           000006  208.253  209.656  206.048  208.253  5.269634e+09  0.000961   
           000009   50.640   51.954   50.158   50.201  1.250000e+10  0.013261   
...                    ...      ...      ...      ...           ...       ...   
2021-12-31 601988    4.412    4.423    4.391    4.402  1.053433e+12  0.002505   
           601989    7.140    7.340    7.100    7.330  4.875183e+10  0.022315   
           601991   17.152   17.337   16.966   17.131  9.777431e+10 -0.021253   
           601998    7.022    7.084    6.859    7.002  2.669881e+11 -0.015882   
           601999   13.665   13.735   13.424   13.474  7.387766e+09 -0.065539   

                   factor_returns  factor_raw_stdd  factor_rank_stdd  \
date       ticker                                                      
2010-01-04 000001        0.024014         1.471805          1.287891   
           000002        0.003206         0.050532          0.187624   
           000005       -0.072381         1.675191          1.651558   
           000006        0.001911         0.321684          0.493383   
           000009        0.009417         0.719442          0.857050   
...                           ...              ...               ...   
2021-12-31 601988       -0.002267         0.371610          0.551291   
           601989        0.004093         1.012247          1.060889   
           601991        0.003606        -0.396719         -0.467903   
           601998        0.011601        -0.223008         -0.220053   
           601999        0.018606        -1.828850         -1.619129   

                   log_market_cap  factor_raw_stdd_neut  factor_rank_stdd_neut  
date       ticker                                                               
2010-01-04 000001       25.003584              1.071015               0.788411  
           000002       25.357171              0.098352               0.258129  
           000005       22.450158              1.539636               1.547097  
           000006       22.385227              0.182034               0.385013  
           000009       23.248995              0.634278               0.800668  
...                           ...                   ...                    ...  
2021-12-31 601988       27.683075              0.155041               0.211992  
           601989       24.610009              1.349233               1.389869  
           601991       25.305928             -0.308392              -0.396257  
           601998       26.310470             -0.507151              -0.623524  
           601999       22.723091             -1.789521              -1.608197  

[4360915 rows x 12 columns]